<a href="https://colab.research.google.com/github/thinkinginmath/ML-tutorials/blob/cursor/tiny-gpt-character-tutorial-9992/Tutorial_Understand_a_Tiny_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This Open in Colab badge uses the review branch `cursor/tiny-gpt-character-tutorial-9992`. It is the link that works before the pull request merges. The copy on `main` is:

https://colab.research.google.com/github/thinkinginmath/ML-tutorials/blob/main/Tutorial_Understand_a_Tiny_GPT.ipynb

That `main` link does not open this notebook until the pull request is merged. Source on the review branch: https://github.com/thinkinginmath/ML-tutorials/blob/cursor/tiny-gpt-character-tutorial-9992/Tutorial_Understand_a_Tiny_GPT.ipynb


# Tutorial: Understand a Tiny GPT by Experimenting with It

## Learning objectives

This notebook trains a very small decoder-only Transformer, the architecture family behind GPT, on characters from a public-domain Shakespeare subset. It is for a reader who has trained a small network before, for example the NumPy MNIST notebook in this repository, and wants to see the pieces of a sequence model with their own eyes.

By the end you will be able to:

1. Turn characters into integer tokens, and tokens into embedding vectors plus position vectors.
2. Read a causal attention mask and say why a position cannot look at future characters.
3. Name the other pieces of a block: several attention heads, a residual connection, layer normalization, and an MLP.
4. Train with next-character cross-entropy on a fixed train/validation split, then sample characters one at a time.
5. Tell an inference-time poke (temperature, cropping the prompt, zeroing positions in a trained model) apart from a comparison that retrains two models under the same budget.

## What this model is

It is a classroom model with about 10^5 parameters. A few hundred CPU steps teach it some local spelling and the look of a speech heading. They do not teach it to write a scene, answer a question, or stand in for a foundation model. Samples below are evidence that the loss moved, not a demo of a useful assistant.

The MNIST notebook, [`Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb`](https://github.com/thinkinginmath/ML-tutorials/blob/main/Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb), trains a 10-way classifier: a digit in, one class out. This notebook trains again, from random weights, but the target is the next character. The loss is still cross-entropy. The new piece is that each prediction may depend only on characters already seen.


## Which small GPT this is

Andrej Karpathy has published several tiny GPT projects. They share an algorithm and they are not the same codebase. This notebook does not copy any of them.

| Project | What it is | Status to keep straight |
| --- | --- | --- |
| [microgpt](http://karpathy.github.io/2026/02/12/microgpt/) (February 2026; [gist](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95)) | About 200 lines of pure Python. Scalar autograd, one transformer block, RMSNorm, ReLU, no biases. Trains on a list of names. | The dependency-free "whole algorithm" sketch. This notebook uses PyTorch instead, so the mask and the training curves are easy to plot. |
| [minGPT](https://github.com/karpathy/minGPT) | A small educational PyTorch GPT. MIT license. | Semi-archived in January 2023. Karpathy pointed readers to nanoGPT for later changes. |
| [nanoGPT](https://github.com/karpathy/nanoGPT) | A PyTorch training repo that can reproduce GPT-2 (124M) and can also run a character-level Shakespeare config. MIT license. | Marked deprecated in November 2025. The README points current training work to [nanochat](https://github.com/karpathy/nanochat). |
| This notebook | An original teaching implementation: explicit causal mask, two matched training runs, and sampling experiments. | Not a GPT-2 reproduction and not a port of microgpt. |

The architecture follows the decoder stack in Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762) (2017), with the pre-norm and GELU choices used by GPT-2 (Radford et al., [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf), 2019). PyTorch can fuse the same mask inside [`torch.nn.functional.scaled_dot_product_attention`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) by passing `is_causal=True` and not also passing `attn_mask`. That function is documented as beta. `nn.Transformer` in PyTorch is an encoder-decoder translation block, which is a different diagram from this decoder-only model. The code below writes the mask out so you can check it.

## Copyright and credits

© Thinking in Math, 2026.

This tutorial text is licensed under the [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License](https://creativecommons.org/licenses/by-nc-sa/4.0/).

- **Author:** thinkinginmath
- **GitHub:** https://github.com/thinkinginmath/ML-tutorials
- **YouTube:** https://www.youtube.com/channel/UC5uV1LcSkHL5zCxKBikdTEg

The text file is a subset of Shakespeare's plays distributed with Karpathy's [char-rnn](https://github.com/karpathy/char-rnn) example data (`data/tinyshakespeare/input.txt`). The plays are in the public domain. The file is about 40,000 lines and is not the complete works: it does not contain *Hamlet*. The char-rnn code is MIT; this notebook does not copy that code. The text is downloaded at runtime and is not committed here.


## Setup and what a CPU run can show

A free Colab CPU runtime is enough. CUDA is used when it exists. Nothing here needs an API key or a paid GPU.

The default run trains two copies of the same small model for 400 steps, then one extra model with a shorter context for a token-matched comparison. On a recent CPU that is on the order of a minute after the text file has downloaded, not an hour. The file is 1,115,394 bytes. Set `TINY_GPT_LONG=1` before starting the kernel for 2,000 steps of the same protocol. Samples get a little more word-like. They remain a toy.

`TINY_GPT_TRAIN_STEPS` overrides the step count. A value of `2` is only a smoke test that the cells run. Do not quote smoke-test losses. The cap is 4,000 steps so a typo cannot start a huge run. The seed is 0. Dropout is 0 so that a rerun on CPU matches and so that a finite-difference gradient check is meaningful. That also means this is not a study of regularization.

Changing `block_size` or turning positions off on a model that has already been trained is an intervention at inference time. It is not a clean statement about what that choice does during learning. The sections below keep those two kinds of experiment apart. Where two losses are compared, the runs share the data split, the optimizer, the batch size, and the step or token budget, and each run has its own generator seeded with 0.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys


def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


def _version_tuple(version):
    parts = []
    for piece in version.split(".")[:3]:
        digits = ""
        for character in piece:
            if character.isdigit():
                digits += character
            else:
                break
        if not digits:
            break
        parts.append(int(digits))
    if not parts:
        raise RuntimeError(f"Could not parse version {version!r}.")
    return tuple(parts)


def _pip_install(arguments):
    print("Installing:", " ".join(arguments))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *arguments])


torch_version = _installed_version("torch")
if torch_version is None:
    print(
        "PyTorch is not installed. Colab already includes it. "
        "Installing a CPU wheel for this local environment only."
    )
    _pip_install(["torch", "--index-url", "https://download.pytorch.org/whl/cpu"])
    torch_version = _installed_version("torch")
if torch_version is None or _version_tuple(torch_version) < (2, 6):
    raise RuntimeError(
        f"Found PyTorch {torch_version}. This notebook expects PyTorch 2.6 or newer "
        "and will not replace a PyTorch install that is already present. "
        "On Colab, use a current runtime. Outside Colab, install a CPU wheel "
        "from https://pytorch.org/get-started/locally/ and rerun from the top."
    )
if _installed_version("matplotlib") is None:
    _pip_install(["matplotlib>=3.7"])
if _installed_version("numpy") is None:
    _pip_install(["numpy>=1.26"])
print("Package check finished.")
print("torch", _installed_version("torch"))
print("matplotlib", _installed_version("matplotlib"))
print("numpy", _installed_version("numpy"))


In [ ]:
import hashlib
import math
import os
import time
import urllib.error
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

%matplotlib inline

SEED = 0
BLOCK_SIZE = 64
BATCH_SIZE = 32
N_EMBD = 64
N_HEAD = 4
N_LAYER = 2
LEARNING_RATE = 3e-3
SHORT_BLOCK = 16
EVAL_BATCHES = 20
MAX_NEW_TOKENS = 180
TEXT_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
TEXT_SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
TEXT_BYTES = 1115394

if N_EMBD % N_HEAD != 0:
    raise ValueError("N_EMBD must be divisible by N_HEAD.")
if SHORT_BLOCK < 2 or SHORT_BLOCK >= BLOCK_SIZE or BLOCK_SIZE % SHORT_BLOCK != 0:
    raise ValueError("SHORT_BLOCK must divide BLOCK_SIZE so the token budgets match.")

long_mode = os.environ.get("TINY_GPT_LONG", "").strip().lower() in {"1", "true", "yes"}
default_steps = 2000 if long_mode else 400
raw_steps = os.environ.get("TINY_GPT_TRAIN_STEPS", "").strip()
if raw_steps:
    try:
        TRAIN_STEPS = int(raw_steps)
    except ValueError as exc:
        raise ValueError(f"TINY_GPT_TRAIN_STEPS must be an integer, got {raw_steps!r}.") from exc
else:
    TRAIN_STEPS = default_steps
if not 1 <= TRAIN_STEPS <= 4000:
    raise ValueError(
        f"TINY_GPT_TRAIN_STEPS must be from 1 to 4000, got {TRAIN_STEPS}. "
        "The default lesson uses 400. TINY_GPT_LONG=1 uses 2000. "
        "A value of 2 is only a smoke test and is not a reported result."
    )

ARTIFACT_DIR = os.environ.get("TINY_GPT_ARTIFACT_DIR", "").strip()
DATA_PATH = Path(os.environ.get("TINY_GPT_DATA_PATH", "data/tinyshakespeare/input.txt"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
print(f"torch {torch.__version__} | device {device}")
if device.type == "cuda":
    print("CUDA:", torch.cuda.get_device_name(0))
else:
    print("Using CPU. A GPU is optional.")
print(
    f"steps {TRAIN_STEPS} | long_mode {long_mode} | block {BLOCK_SIZE} | "
    f"batch {BATCH_SIZE} | width {N_EMBD} | heads {N_HEAD} | layers {N_LAYER}"
)
print("Dropout is 0. The seed is 0.")


## The text

The download is the Tiny Shakespeare file from the char-rnn repository, checked against a size and SHA-256 recorded from that URL. A mismatch stops the notebook. Another corpus is not swapped in.

The split is the first 90% of characters for training and the last 10% for validation. It is a cut through one file, not a shuffle, and not a promise about a new author. Validation lines are later lines of the same subset, so the character set overlaps and some phrases can resemble the training text. The number you will see is next-character cross-entropy on this cut.

The vocabulary is the set of characters that occur in the training cut. A prompt character outside that set is an error. The notebook does not replace it with a stand-in.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_text(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    last_error = None
    for attempt in range(1, 4):
        try:
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "thinkinginmath-ml-tutorials-tiny-gpt"},
            )
            with urllib.request.urlopen(request, timeout=60) as response:
                status = getattr(response, "status", 200)
                if status != 200:
                    raise RuntimeError(f"HTTP {status} for {url}")
                payload = response.read()
            if len(payload) != TEXT_BYTES:
                raise RuntimeError(f"Expected {TEXT_BYTES} bytes and received {len(payload)} from {url}.")
            temporary.write_bytes(payload)
            temporary.replace(destination)
            return destination
        except (urllib.error.URLError, TimeoutError, RuntimeError, OSError) as exc:
            last_error = exc
            print(f"Download attempt {attempt} failed: {exc}")
            if temporary.exists():
                temporary.unlink()
    raise RuntimeError(
        f"Could not download the Shakespeare subset from {url}. Last error: {last_error}. "
        "This notebook stops instead of substituting a different text. "
        "Check the network and https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt ."
    ) from last_error


if DATA_PATH.exists() and DATA_PATH.stat().st_size == TEXT_BYTES and sha256_file(DATA_PATH) == TEXT_SHA256:
    print("Reusing the text file that already matches the checksum.")
else:
    if DATA_PATH.exists():
        DATA_PATH.unlink()
    print(f"Downloading {TEXT_BYTES} bytes from {TEXT_URL}")
    download_text(TEXT_URL, DATA_PATH)
if sha256_file(DATA_PATH) != TEXT_SHA256 or DATA_PATH.stat().st_size != TEXT_BYTES:
    raise RuntimeError(
        "The text file does not match the expected size or SHA-256 "
        f"({TEXT_SHA256}). Refusing to train on it."
    )
text = DATA_PATH.read_text(encoding="utf-8")
if len(text) != TEXT_BYTES:
    raise RuntimeError(
        f"Decoded length {len(text)} does not equal the byte length {TEXT_BYTES}. "
        "This lesson expects the ASCII Tiny Shakespeare file."
    )
if not text.startswith("First Citizen:"):
    raise RuntimeError("The file does not start with the expected Tiny Shakespeare line.")
print(f"Characters: {len(text):,}. Distinct characters in the whole file: {len(set(text))}.")
print("Contains the name Hamlet:", "Hamlet" in text)


## Tokens, embeddings, and positions

A character-level tokenizer is a sorted list of characters. `encode` maps a string to integers. `decode` maps integers back. There is no byte-pair vocabulary and no special beginning-of-sentence token. An empty prompt starts from a newline, which is how the file separates lines. A prompt longer than the context window keeps its last `block_size` characters. That crop happens at sampling time. It does not change the model that was trained.

Each token id becomes a learned vector. Each position in the window, 0 through `block_size - 1`, becomes another learned vector of the same width. The block sees their sum. Without the position vectors, the attention pattern can still tell tokens apart by their content, and it cannot tell order apart except through the causal mask's "already seen" rule. The training section measures that claim by training a second model that skips the addition. Zeroing the position vectors of a model that was trained with them is a different, later experiment.


In [ ]:
# TINY_GPT_PURE_HELPERS
import torch


def split_characters(text, train_fraction=0.9):
    if not isinstance(text, str) or len(text) < 100:
        raise ValueError("Need a text of at least 100 characters to split.")
    if not isinstance(train_fraction, float) or not 0.5 <= train_fraction < 1.0:
        raise ValueError("train_fraction must be a float in [0.5, 1).")
    cut = int(len(text) * train_fraction)
    if cut < 10 or cut >= len(text):
        raise ValueError("The split left one side empty.")
    return text[:cut], text[cut:]


def unknown_characters(text, alphabet):
    allowed = set(alphabet)
    return sorted({character for character in text if character not in allowed})


def encode_characters(text, stoi):
    missing = unknown_characters(text, stoi.keys())
    if missing:
        shown = ", ".join(repr(character) for character in missing)
        raise ValueError(
            "Prompt contains characters that are not in the training vocabulary: "
            f"{shown}. This notebook does not substitute a replacement character."
        )
    if text == "":
        raise ValueError("Refusing to encode an empty string. Choose a documented start character first.")
    return [stoi[character] for character in text]


def causal_allowed(query_len, key_len, device=None):
    if query_len < 1 or key_len < 1:
        raise ValueError("Attention lengths must be positive.")
    query_index = torch.arange(query_len, device=device)[:, None]
    key_index = torch.arange(key_len, device=device)[None, :]
    return key_index <= query_index


def attention_weights(scores, causal):
    if scores.ndim < 2:
        raise ValueError("scores must end with (query, key).")
    query_len, key_len = scores.shape[-2], scores.shape[-1]
    working = scores
    if causal:
        allowed = causal_allowed(query_len, key_len, device=scores.device)
        while allowed.ndim < scores.ndim:
            allowed = allowed.unsqueeze(0)
        working = scores.masked_fill(~allowed, float("-inf"))
    weights = torch.softmax(working, dim=-1)
    if not torch.isfinite(weights).all():
        raise RuntimeError(
            "Attention weights are not finite. A causal row may have had no allowed key."
        )
    return weights


def validate_temperature(temperature):
    if isinstance(temperature, bool) or not isinstance(temperature, (int, float)):
        raise TypeError(f"temperature must be a real number, got {type(temperature).__name__}.")
    value = float(temperature)
    if value < 0.0 or value != value:
        raise ValueError(f"temperature must be >= 0, got {temperature!r}.")
    return value


def crop_context(token_row, block_size):
    if token_row.ndim != 2 or token_row.shape[0] != 1:
        raise ValueError("token_row must have shape (1, time).")
    if token_row.shape[1] < 1:
        raise ValueError("Cannot crop an empty context.")
    if not isinstance(block_size, int) or isinstance(block_size, bool) or block_size < 1:
        raise ValueError(f"block_size must be a positive integer, got {block_size!r}.")
    return token_row[:, -block_size:]


def sample_next_id(logits, temperature, generator):
    if logits.ndim != 2:
        raise ValueError("logits must have shape (batch, vocab).")
    temperature = validate_temperature(temperature)
    if temperature == 0.0:
        return torch.argmax(logits, dim=-1, keepdim=True)
    probabilities = torch.softmax(logits / temperature, dim=-1)
    if not torch.isfinite(probabilities).all() or torch.any(probabilities < 0):
        raise RuntimeError("The sampling distribution is not a valid probability vector.")
    return torch.multinomial(probabilities, num_samples=1, generator=generator)


demo_scores = torch.zeros(4, 4)
demo_causal = attention_weights(demo_scores, causal=True)
demo_open = attention_weights(demo_scores, causal=False)
print("Causal weights when every score is 0 (uniform over the allowed past):")
print(demo_causal)
print("Without the mask, the same scores attend everywhere:")
print(demo_open[0])


In [ ]:
train_text, val_text = split_characters(text, 0.9)
alphabet = sorted(set(train_text))
missing_in_val = unknown_characters(val_text, alphabet)
if missing_in_val:
    shown = ", ".join(repr(character) for character in missing_in_val)
    raise RuntimeError(
        "Validation text contains characters absent from the training cut: "
        f"{shown}. Refusing to drop or replace them."
    )
stoi = {character: index for index, character in enumerate(alphabet)}
itos = {index: character for character, index in stoi.items()}
vocab_size = len(alphabet)
train_ids = torch.tensor(encode_characters(train_text, stoi), dtype=torch.long)
val_ids = torch.tensor(encode_characters(val_text, stoi), dtype=torch.long)
if train_ids.numel() <= BLOCK_SIZE or val_ids.numel() <= BLOCK_SIZE:
    raise RuntimeError("The split is shorter than the context window.")
print(f"Vocabulary ({vocab_size}): {''.join(alphabet)!r}")
print(f"Train characters {train_ids.numel():,} | validation characters {val_ids.numel():,}")
print("Uniform-guess cross-entropy would be", round(math.log(vocab_size), 4), "nats.")
print("--- first training lines ---")
print(train_text[:280])
print("--- first validation lines ---")
print(val_text[:280])


## Causal self-attention

For one head, queries, keys, and values are linear projections of the current hidden vectors. The score of query position $t$ with key position $s$ is

$$
\frac{q_t \cdot k_s}{\sqrt{d}}.
$$

The causal mask adds $-\infty$ wherever $s > t$, then softmax turns each row into weights that sum to 1. Multiplying those weights by the values is a weighted average of earlier positions only, including the current one. Several heads do this in parallel on slices of the width, and a final linear layer mixes them.

The cell above already printed the mask on an all-zero score matrix. Row $t$ is uniform on columns $0,\ldots,t$ and zero after that. The model calls the same `attention_weights` function, so the picture and the training path are the same rule.

An attention map later in the notebook is a picture of these weights for one head and one window. It is not, by itself, a proof of why a character was sampled. The weights are one layer of one head, before the rest of the network mixes them.


## The rest of the block

Each block is pre-norm, as in GPT-2: layer normalization, then the sublayer, then a residual add. The second sublayer is an MLP that expands the width by 4, applies GELU (`nn.GELU`, the error-function version), and projects back. A residual add means the block can leave the hidden vector unchanged if that is useful. After the last block, one more layer norm and a linear head produce a logit for every character in the vocabulary. The target at position $t$ is the token at position $t+1$. Every position in the window has a target, so the cross-entropy is a plain mean. There is no padding index to ignore.

The head is not tied to the token embedding. Tying is a common GPT trick. Leaving the head separate makes the prediction layer visible.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must be divisible by n_head.")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.query_key_value = nn.Linear(n_embd, 3 * n_embd)
        self.output_projection = nn.Linear(n_embd, n_embd)

    def forward(self, hidden, return_weights=False):
        batch, time, channels = hidden.shape
        projected = self.query_key_value(hidden)
        projected = projected.reshape(batch, time, 3, self.n_head, self.head_dim)
        query, key, value = projected.unbind(dim=2)
        query = query.transpose(1, 2)
        key = key.transpose(1, 2)
        value = value.transpose(1, 2)
        scores = (query @ key.transpose(-2, -1)) * (self.head_dim ** -0.5)
        weights = attention_weights(scores, causal=True)
        mixed = (weights @ value).transpose(1, 2).contiguous().reshape(batch, time, channels)
        output = self.output_projection(mixed)
        if return_weights:
            return output, weights
        return output


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.attention_norm = nn.LayerNorm(n_embd)
        self.attention = CausalSelfAttention(n_embd, n_head, block_size)
        self.mlp_norm = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, hidden, return_weights=False):
        if return_weights:
            attended, weights = self.attention(self.attention_norm(hidden), return_weights=True)
            hidden = hidden + attended
            hidden = hidden + self.mlp(self.mlp_norm(hidden))
            return hidden, weights
        hidden = hidden + self.attention(self.attention_norm(hidden))
        hidden = hidden + self.mlp(self.mlp_norm(hidden))
        return hidden


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer, use_positions=True):
        super().__init__()
        self.block_size = block_size
        self.use_positions = use_positions
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        # Always allocated so two models seeded the same way share later initial weights.
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(
            TransformerBlock(n_embd, n_head, block_size) for _ in range(n_layer)
        )
        self.final_norm = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, token_ids, targets=None, position_scale=1.0, return_first_attention=False):
        batch, time = token_ids.shape
        if time < 1 or time > self.block_size:
            raise ValueError(
                f"Sequence length must be from 1 to {self.block_size}, got {time}. "
                "Crop the context before calling the model. Do not grow a trained position table."
            )
        hidden = self.token_embedding(token_ids)
        if self.use_positions:
            positions = torch.arange(time, device=token_ids.device)
            hidden = hidden + position_scale * self.position_embedding(positions)
        elif position_scale != 1.0:
            raise ValueError(
                "position_scale is only defined for a model that was built with positions. "
                "A model trained without positions does not have a position pathway to rescale."
            )
        first_weights = None
        for index, block in enumerate(self.blocks):
            if return_first_attention and index == 0:
                hidden, first_weights = block(hidden, return_weights=True)
            else:
                hidden = block(hidden)
        logits = self.lm_head(self.final_norm(hidden))
        loss = None
        if targets is not None:
            if targets.shape != token_ids.shape:
                raise ValueError("targets must have the same shape as token_ids.")
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, first_weights


def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())


torch.manual_seed(SEED)
probe = TinyGPT(vocab_size, BLOCK_SIZE, N_EMBD, N_HEAD, N_LAYER, use_positions=True).to(device)
print(f"Parameters: {parameter_count(probe):,}")
print(f"Logit width {vocab_size}, context cap {probe.block_size}, position pathway {probe.use_positions}")


## Checks before training

Three checks, all on a freshly initialized model, before any claim about Shakespeare.

1. **Causal mask.** Change the last token of a batch. Logits at earlier positions must stay the same, because those positions cannot attend to the last token. The last position's logits should change, because that position can see its own token. The pure function above is what the module calls. Turning the mask off in that function lets probability leak into future columns. That second print is the mask's job, not a second trained model.

2. **Gradients exist.** One backward pass should put a finite, nonzero gradient on the token embedding, the attention projection, the MLP, and the output head. The position embedding is in the graph, so it should receive a gradient too.

3. **One finite-difference check.** For the first bias entry of the output head, compare the analytic gradient with a central difference. This is one scalar, not a proof about every weight. Dropout is 0 and the model is in eval mode so the two sides see the same function. A failure here means the loss is not the function the optimizer will differentiate.


In [ ]:
def get_batch(split_ids, block_size, batch_size, generator):
    if split_ids.numel() <= block_size:
        raise ValueError(
            f"Need more than {block_size} tokens to draw a window, found {split_ids.numel()}."
        )
    starts = torch.randint(
        0,
        split_ids.numel() - block_size,
        (batch_size,),
        generator=generator,
    )
    inputs = torch.stack([split_ids[start:start + block_size] for start in starts.tolist()])
    targets = torch.stack([split_ids[start + 1:start + block_size + 1] for start in starts.tolist()])
    return inputs, targets


probe.eval()
generator = torch.Generator(device="cpu").manual_seed(SEED)
window, _targets = get_batch(train_ids, 16, 2, generator)
window = window.to(device)
changed = window.clone()
changed[:, -1] = (changed[:, -1] + 1) % vocab_size
with torch.inference_mode():
    original_logits, _, _ = probe(window)
    changed_logits, _, _ = probe(changed)
past_matches = torch.allclose(original_logits[:, :-1], changed_logits[:, :-1], atol=1e-5, rtol=1e-5)
last_moves = not torch.allclose(original_logits[:, -1], changed_logits[:, -1], atol=1e-4, rtol=1e-4)
print("Earlier logits unchanged when the last token changes:", bool(past_matches))
print("Last-position logits do change:", bool(last_moves))
if not past_matches or not last_moves:
    raise RuntimeError(
        "The causal-mask sanity check failed. Earlier positions must ignore a future token, "
        "and the edited position must be able to see its own token."
    )

open_weights = attention_weights(torch.zeros(4, 4), causal=False)
closed_weights = attention_weights(torch.zeros(4, 4), causal=True)
future = torch.triu(torch.ones(4, 4), diagonal=1).bool()
if torch.any(closed_weights[future] > 1e-6):
    raise RuntimeError("Causal attention put weight on a future column.")
if float(open_weights[future].sum()) <= 0:
    raise RuntimeError("The unmasked demo unexpectedly put no weight on future columns.")
print("Unmasked future mass:", round(float(open_weights[future].sum()), 4))
print("Causal future mass:", round(float(closed_weights[future].sum()), 6))

probe.train()
probe.zero_grad(set_to_none=True)
batch_inputs, batch_targets = get_batch(train_ids, 16, 4, torch.Generator().manual_seed(1))
_logits, loss, _weights = probe(batch_inputs.to(device), batch_targets.to(device))
loss.backward()
watched = {
    "token_embedding": probe.token_embedding.weight.grad,
    "position_embedding": probe.position_embedding.weight.grad,
    "attention": probe.blocks[0].attention.query_key_value.weight.grad,
    "mlp": probe.blocks[0].mlp[0].weight.grad,
    "lm_head": probe.lm_head.weight.grad,
}
for name, gradient in watched.items():
    if gradient is None or not torch.isfinite(gradient).all() or float(gradient.abs().sum()) == 0.0:
        raise RuntimeError(f"{name} did not receive a finite nonzero gradient.")
    print(f"{name} grad abs sum {float(gradient.abs().sum()):.4f}")

probe.eval()
probe.zero_grad(set_to_none=True)
held_inputs = batch_inputs.to(device)
held_targets = batch_targets.to(device)
_logits, held_loss, _weights = probe(held_inputs, held_targets)
held_loss.backward()
analytic = float(probe.lm_head.bias.grad[0].detach().cpu())
epsilon = 1e-3
with torch.no_grad():
    original_bias = probe.lm_head.bias.detach().clone()
    probe.lm_head.bias[0] = original_bias[0] + epsilon
    loss_plus = float(probe(held_inputs, held_targets)[1].detach().cpu())
    probe.lm_head.bias.copy_(original_bias)
    probe.lm_head.bias[0] = original_bias[0] - epsilon
    loss_minus = float(probe(held_inputs, held_targets)[1].detach().cpu())
    probe.lm_head.bias.copy_(original_bias)
numeric = (loss_plus - loss_minus) / (2 * epsilon)
absolute_gap = abs(numeric - analytic)
print(f"Output-bias finite difference {numeric:.6f} | backprop {analytic:.6f} | gap {absolute_gap:.6e}")
if absolute_gap > 1e-3:
    raise RuntimeError(
        "The finite-difference check on lm_head.bias[0] disagrees with backprop. "
        "Training would optimize a different function from the one printed as the loss."
    )
print("Pre-training checks passed.")


## Train two models for the same number of steps

Both models are constructed after `torch.manual_seed(0)`, and both allocate a position table, so the token embeddings and the blocks start from the same weights. One model adds the position vectors. The other does not. Each has its own AdamW optimizer at learning rate `3e-3` and its own batch generator seeded with 0. The context length, batch size, and step count match, so the windows match as well. Gradient norms are clipped at 1. That clip is part of both runs.

The curve is the loss of each training batch. It jumps around. The number to quote is the mean loss on 20 fresh validation batches after training, with the model in eval mode and a generator seeded with 123. A uniform guess over the vocabulary is $\log V$ nats. Beating that only means the model is better than chance on this file.

`TINY_GPT_TRAIN_STEPS=2` still runs this cell. Those losses are a code-path check.


In [ ]:
def train_model(model, steps, block_size, seed):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    generator = torch.Generator(device="cpu").manual_seed(seed)
    model.train()
    history = []
    started = time.perf_counter()
    for step in range(1, steps + 1):
        inputs, targets = get_batch(train_ids, block_size, BATCH_SIZE, generator)
        _logits, loss, _weights = model(inputs.to(device), targets.to(device))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        history.append(float(loss.detach().cpu()))
        if step == 1 or step % 50 == 0 or step == steps:
            print(f"step {step}/{steps} batch train loss {history[-1]:.3f}")
    seconds = time.perf_counter() - started
    return history, seconds


def estimate_loss(model, split_ids, block_size, batches, seed):
    was_training = model.training
    model.eval()
    generator = torch.Generator(device="cpu").manual_seed(seed)
    losses = []
    with torch.inference_mode():
        for _ in range(batches):
            inputs, targets = get_batch(split_ids, block_size, BATCH_SIZE, generator)
            _logits, loss, _weights = model(inputs.to(device), targets.to(device))
            losses.append(float(loss.detach().cpu()))
    if was_training:
        model.train()
    return sum(losses) / len(losses)


def build_model(use_positions, block_size, seed):
    torch.manual_seed(seed)
    model = TinyGPT(
        vocab_size,
        block_size,
        N_EMBD,
        N_HEAD,
        N_LAYER,
        use_positions=use_positions,
    ).to(device)
    return model


model_with = build_model(True, BLOCK_SIZE, SEED)
model_without = build_model(False, BLOCK_SIZE, SEED)
same_tokens = torch.equal(model_with.token_embedding.weight, model_without.token_embedding.weight)
same_head = torch.equal(model_with.lm_head.weight, model_without.lm_head.weight)
print("Token embeddings match at init:", bool(same_tokens))
print("Output head matches at init:", bool(same_head))
if not same_tokens or not same_head:
    raise RuntimeError(
        "The two position settings did not start from the same weights. "
        "A loss gap would then mix initialization with the position flag."
    )

print("Training with position embeddings.")
history_with, seconds_with = train_model(model_with, TRAIN_STEPS, BLOCK_SIZE, SEED)
print("Training without adding position embeddings.")
history_without, seconds_without = train_model(model_without, TRAIN_STEPS, BLOCK_SIZE, SEED)
model_with.eval()
model_without.eval()
val_with = estimate_loss(model_with, val_ids, BLOCK_SIZE, EVAL_BATCHES, seed=123)
val_without = estimate_loss(model_without, val_ids, BLOCK_SIZE, EVAL_BATCHES, seed=123)
train_with = estimate_loss(model_with, train_ids, BLOCK_SIZE, EVAL_BATCHES, seed=123)
train_without = estimate_loss(model_without, train_ids, BLOCK_SIZE, EVAL_BATCHES, seed=123)
chance = math.log(vocab_size)
print(
    f"With positions: train {train_with:.3f} | validation {val_with:.3f} | "
    f"{seconds_with:.1f}s | {TRAIN_STEPS} steps"
)
print(
    f"Without positions: train {train_without:.3f} | validation {val_without:.3f} | "
    f"{seconds_without:.1f}s | {TRAIN_STEPS} steps"
)
print(f"Chance (uniform over {vocab_size} characters): {chance:.3f}")
print(
    "These validation numbers use the same 20-batch generator seed. "
    "They are a small-budget comparison on this split, not a general law about position embeddings."
)

figure, axis = plt.subplots(figsize=(7.5, 4.2))
axis.plot(history_with, label="train batch, with positions", color="#0f766e")
axis.plot(history_without, label="train batch, positions not added", color="#b45309")
axis.axhline(val_with, color="#0f766e", linestyle="--", label=f"final val with positions {val_with:.3f}")
axis.axhline(val_without, color="#b45309", linestyle="--", label=f"final val without {val_without:.3f}")
axis.axhline(chance, color="#64748b", linestyle=":", label=f"uniform chance {chance:.3f}")
axis.set_xlabel("training step")
axis.set_ylabel("cross-entropy (nats)")
axis.set_title("Same initialization, steps, and windows. One flag differs.")
axis.legend(fontsize=8)
figure.tight_layout()
if ARTIFACT_DIR:
    destination = Path(ARTIFACT_DIR) / "tiny_gpt_loss_curves.png"
    destination.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(destination, dpi=120, bbox_inches="tight")
    print("Saved figure tiny_gpt_loss_curves.png")
display(figure)
plt.close(figure)


## Generate, and turn the temperature knob

Sampling starts from a fixed slice of the **validation** text, so the prompt is not a training window. Each new id is drawn from the distribution at the last position, appended, and the window is cropped back to `block_size` characters. Temperature divides the logits before softmax.

Temperature 0 is greedy: always take the argmax. It does not use the random generator. A positive temperature uses a generator seeded with 0, on the same device as the logits. A negative temperature or a character outside the training alphabet raises an error. The cap on new characters in this lesson is 400.

Temperature is an inference setting on one trained model. It does not retrain anything. A higher temperature spreads probability. The samples are still next-character draws from a tiny model. Reading them as prose overstates what the loss measured.


In [ ]:
def ids_from_prompt(prompt):
    if prompt == "":
        if "\n" not in stoi:
            raise ValueError("The training alphabet has no newline, so an empty prompt has no documented start.")
        prompt = "\n"
    encoded = encode_characters(prompt, stoi)
    return torch.tensor([encoded], dtype=torch.long, device=device), prompt


@torch.inference_mode()
def generate(model, prompt, max_new_tokens, temperature, seed, position_scale=1.0):
    if not isinstance(max_new_tokens, int) or isinstance(max_new_tokens, bool) or not 1 <= max_new_tokens <= 400:
        raise ValueError(f"max_new_tokens must be an integer from 1 to 400, got {max_new_tokens!r}.")
    temperature = validate_temperature(temperature)
    token_row, used_prompt = ids_from_prompt(prompt)
    generator = torch.Generator(device=token_row.device).manual_seed(seed)
    model.eval()
    for _ in range(max_new_tokens):
        context = crop_context(token_row, model.block_size)
        logits, _loss, _weights = model(context, position_scale=position_scale)
        next_id = sample_next_id(logits[:, -1, :], temperature, generator)
        token_row = torch.cat([token_row, next_id], dim=1)
    produced = "".join(itos[int(index)] for index in token_row[0].tolist())
    return used_prompt, produced


def distribution_entropy(model, prompt, temperature):
    temperature = validate_temperature(temperature)
    if temperature == 0.0:
        raise ValueError("Entropy is for a positive temperature. Greedy sampling does not use a distribution average.")
    token_row, _prompt = ids_from_prompt(prompt)
    context = crop_context(token_row, model.block_size)
    with torch.inference_mode():
        logits, _loss, _weights = model(context)
        probabilities = torch.softmax(logits[0, -1, :] / temperature, dim=-1)
    log_probabilities = torch.log(probabilities.clamp_min(1e-12))
    return float(-(probabilities * log_probabilities).sum().cpu())


def validation_excerpt(length):
    marker = "\nBAPTISTA:\n"
    index = val_text.find(marker)
    if index < 0 or index + 1 + length > len(val_text):
        raise RuntimeError(
            "Could not find a BAPTISTA line in the validation text long enough for this prompt."
        )
    return val_text[index + 1:index + 1 + length]


prompt = validation_excerpt(48)
print("Validation prompt:")
print(prompt)
print("--- temperature 0 (greedy) ---")
_used, greedy = generate(model_with, prompt, MAX_NEW_TOKENS, temperature=0.0, seed=0)
print(greedy)
print("--- temperature 0.8 ---")
_used, mild = generate(model_with, prompt, MAX_NEW_TOKENS, temperature=0.8, seed=0)
print(mild)
print("--- temperature 1.5 ---")
_used, hot = generate(model_with, prompt, MAX_NEW_TOKENS, temperature=1.5, seed=0)
print(hot)
for temperature in (0.5, 0.8, 1.5):
    entropy = distribution_entropy(model_with, prompt, temperature)
    print(f"Next-token entropy at temperature {temperature}: {entropy:.3f} nats")
print("Entropy is a property of the next-token distribution, not a score for the paragraph.")

try:
    generate(model_with, prompt, 10, temperature=-1, seed=0)
    raise RuntimeError("A negative temperature should have been rejected.")
except ValueError as exc:
    print("Negative temperature rejected:", exc)
try:
    encode_characters(prompt + "\u2603", stoi)
    raise RuntimeError("An unknown character should have been rejected.")
except ValueError as exc:
    print("Unknown character rejected:", exc)
long_prompt = val_text[: BLOCK_SIZE + 30]
cropped = crop_context(torch.tensor([encode_characters(long_prompt, stoi)]), BLOCK_SIZE)
if cropped.shape[1] != BLOCK_SIZE:
    raise RuntimeError("Long prompts were not cropped to the context window.")
print(f"A prompt of {len(long_prompt)} characters is cropped to {cropped.shape[1]} before the forward pass.")

if ARTIFACT_DIR:
    sample_path = Path(ARTIFACT_DIR) / "tiny_gpt_samples.txt"
    sample_path.parent.mkdir(parents=True, exist_ok=True)
    sample_path.write_text(
        "PROMPT\n" + prompt + "\n\nGREEDY\n" + greedy + "\n\nTEMPERATURE 0.8\n" + mild + "\n\nTEMPERATURE 1.5\n" + hot + "\n",
        encoding="utf-8",
    )
    print("Saved tiny_gpt_samples.txt")


## Positions: a retrained comparison and a broken-model intervention

The validation losses printed above are the controlled comparison. Same initialization, same windows, same number of steps. One model was trained without adding position vectors. Read the gap with the step count in mind. A few hundred steps on one file can move the loss and can also be noisy. It is not a sweep over datasets.

The next lines do something else. They take the model that **was** trained with positions and multiply those vectors by 0 at inference time. The weights in the attention layers were learned while positions were present, so they are the wrong weights for an input that suddenly has none. The sample can get worse, and that worsening is not the same number as the retrained comparison. If the two stories disagreed, the retrained pair would be the one about learning. The zeroed forward pass would still only be a broken input.


In [ ]:
print(
    f"Retrained validation loss with positions {val_with:.3f}, "
    f"without positions {val_without:.3f}, over {TRAIN_STEPS} matched steps."
)
_used, broken = generate(
    model_with,
    prompt,
    120,
    temperature=0.0,
    seed=0,
    position_scale=0.0,
)
print("Greedy sample from the with-positions model:")
print(greedy[:240])
print("Same weights, positions multiplied by 0 at inference, greedy:")
print(broken[:240])
print(
    "The second sample is an inference intervention. "
    "The loss pair above is the matched retraining."
)


## Context length

The trained model accepts any window up to `block_size` because the mask is built for the current length. Feeding it 16 characters instead of 64 is a different prompt. Greedy samples from those two prompts can differ. That difference does not say what would have happened if you had trained with a context of 16.

A training comparison has to deal with two extra facts. A shorter window sees fewer tokens per step, so the same step count is a smaller token budget. And the position table has a different shape, so the initial weights are not an identical copy with one integer changed. The cell below trains a second model with context 16 for four times as many steps, which matches the token count of the 64-character run (`64 / 16 = 4`, batch size unchanged). Both models are then scored on validation windows of length 16, the longest context the smaller model can take. The 64-character model is the one already trained above. It is not trained again for this table.

Per-token loss at context 16 is a different prediction problem from loss at context 64. The table does not pick a winner across those two tasks. It only asks, on the shared task of 16-character windows, how the token-matched short-context training compares with the model that was trained at 64 and then evaluated at 16.


In [ ]:
context_steps = TRAIN_STEPS * (BLOCK_SIZE // SHORT_BLOCK)
print(
    f"Short-context run: block {SHORT_BLOCK}, steps {context_steps}, "
    f"batch {BATCH_SIZE}. Token count matches {TRAIN_STEPS} steps at block {BLOCK_SIZE}."
)
model_short = build_model(True, SHORT_BLOCK, SEED)
history_short, seconds_short = train_model(model_short, context_steps, SHORT_BLOCK, SEED)
model_short.eval()
val_short_at_16 = estimate_loss(model_short, val_ids, SHORT_BLOCK, EVAL_BATCHES, seed=321)
val_long_at_16 = estimate_loss(model_with, val_ids, SHORT_BLOCK, EVAL_BATCHES, seed=321)
print(f"Validation loss at context {SHORT_BLOCK}, seed 321, {EVAL_BATCHES} batches:")
print(f"  trained at {BLOCK_SIZE}, evaluated at {SHORT_BLOCK}: {val_long_at_16:.3f}")
print(f"  trained at {SHORT_BLOCK} for a matched token budget: {val_short_at_16:.3f} ({seconds_short:.1f}s)")
print("Do not compare these two numbers with the context-64 validation loss above. That was a longer window.")

full_prompt = validation_excerpt(BLOCK_SIZE)
short_prompt = full_prompt[-SHORT_BLOCK:]
_used, from_full = generate(model_with, full_prompt, 80, temperature=0.0, seed=0)
_used, from_short = generate(model_with, short_prompt, 80, temperature=0.0, seed=0)
print("Greedy continuation, same trained model, prompt length", BLOCK_SIZE)
print(from_full)
print("Greedy continuation, same weights, only the last", SHORT_BLOCK, "prompt characters")
print(from_short)
print("Cropping the prompt did not retrain the position table or the mask size.")


## Look at one attention map

The heatmap is head 0 of block 0 on a 24-character slice of the validation prompt, after training. Each row is a query character. Pale cells are future positions the mask set to zero weight. A bright cell means that head put weight on that earlier character for that query.

This is a weight, after softmax, in one head. Later layers and the output head still mix information. Do not read a bright cell as the reason the sampler picked a letter.


In [ ]:
attention_text = validation_excerpt(24)
attention_ids, _used = ids_from_prompt(attention_text)
with torch.inference_mode():
    _logits, _loss, weights = model_with(attention_ids, return_first_attention=True)
if weights is None:
    raise RuntimeError("The model did not return first-block attention weights.")
head = weights[0, 0].detach().cpu()
if head.shape != (len(attention_text), len(attention_text)):
    raise RuntimeError(f"Unexpected attention shape {tuple(head.shape)}.")
future = torch.triu(torch.ones(head.shape[0], head.shape[1]), diagonal=1).bool()
row_sums = head.sum(dim=-1)
if not torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4):
    raise RuntimeError("Attention rows do not sum to 1.")
if torch.any(head[future] > 1e-5):
    raise RuntimeError("The plotted head put weight on a future character.")
print(f"Max weight on a future cell: {float(head[future].max()) if future.any() else 0:.3e}")

labels = [repr(character)[1:-1] for character in attention_text]
figure, axis = plt.subplots(figsize=(7.2, 6.2))
image = axis.imshow(head.numpy(), cmap="Blues", vmin=0.0, vmax=max(float(head.max()), 1e-6))
axis.set_xticks(range(len(labels)))
axis.set_yticks(range(len(labels)))
axis.set_xticklabels(labels, fontsize=7)
axis.set_yticklabels(labels, fontsize=7)
axis.set_xlabel("key (earlier characters, and the query itself)")
axis.set_ylabel("query")
axis.set_title("Block 0, head 0. Pale future cells are the causal mask.")
figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
figure.tight_layout()
if ARTIFACT_DIR:
    destination = Path(ARTIFACT_DIR) / "tiny_gpt_attention.png"
    destination.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(destination, dpi=120, bbox_inches="tight")
    print("Saved figure tiny_gpt_attention.png")
display(figure)
plt.close(figure)


## Limitations

The default CPU budget produces letter salad with occasional English-looking fragments. A longer run (`TINY_GPT_LONG=1`) usually lowers the loss and still does not produce a scene you should trust. Nothing here was aligned, instruction-tuned, or evaluated as an assistant.

The validation cut is the tail of one Shakespeare subset. It shares characters, spellings, and stage directions with the training cut. Beating uniform chance on that tail is a weak statement. It is not a result about new plays, new languages, or factual questions.

Attention plots and temperature samples are descriptions of one forward pass. The position-embedding loss pair is a matched retrain at this width, depth, learning rate, and step count. A different width could move the gap. The context-16 table matches tokens and then scores both models at context 16. It does not say that a shorter context is a better or worse way to write.

`nn.Transformer` and a pretrained GPT-2 checkpoint are different tools. This file trains its own small weights and deletes nothing you could deploy, because there is nothing here to deploy. Weights and the text file stay in the runtime directory.


## Run it again

Unset `TINY_GPT_TRAIN_STEPS` and `TINY_GPT_LONG` for the 400-step lesson. Export `TINY_GPT_LONG=1` before the kernel starts for 2,000 steps. Export `TINY_GPT_TRAIN_STEPS=2` only to confirm that every cell executes.

The MNIST training tutorial remains [Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb](https://github.com/thinkinginmath/ML-tutorials/blob/main/Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb). This file does not modify it.
